In [3]:
# adjust_ssb_frequency.ipynb

# 必要套件
import re
import difflib
from typing import Tuple
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain.prompts import PromptTemplate
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from pathlib import Path
# 可根據 3GPP TS 38.104 轉換 frequency <-> ARFCN
# 以下簡化版僅處理 FR1

def freq_to_arfcn(freq_hz: float) -> int:
    freq_mhz = freq_hz / 1e6
    if 410 <= freq_mhz <= 526:  # n5
        return int((freq_mhz - 410) / 0.005 + 173800)
    elif 3000 <= freq_mhz <= 5000:  # n78
        return int((freq_mhz - 3000) / 0.015 + 620000)
    else:
        raise ValueError("Unsupported frequency range for ARFCN conversion")


def parse_freq_from_query(query: str) -> float:
    pattern = r"(\d+\.?\d*)\s*(GHz|MHz|kHz|Hz)"
    match = re.search(pattern, query, re.IGNORECASE)
    if not match:
        raise ValueError("No frequency found in query")

    value, unit = float(match.group(1)), match.group(2).lower()
    multiplier = {
        "ghz": 1e9,
        "mhz": 1e6,
        "khz": 1e3,
        "hz": 1,
    }[unit]
    return value * multiplier


def extract_ssb_lines(du_conf: str) -> list:
    lines = du_conf.splitlines()
    ssb_lines = [l for l in lines if 'ssb' in l.lower() or 'absoluteFrequencySSB' in l]
    return ssb_lines


def modify_absolute_frequency_ssb(du_conf: str, new_freq_arfcn: int) -> Tuple[str, str]:
    pattern = r"(absoluteFrequencySSB\s*=\s*)(\d+)"
    new_conf, count = re.subn(pattern, f"\\g<1>{new_freq_arfcn}", du_conf)
    if count == 0:
        raise ValueError("Could not find 'absoluteFrequencySSB' to modify")

    diff = '\n'.join(difflib.unified_diff(
        du_conf.splitlines(),
        new_conf.splitlines(),
        lineterm='',
        fromfile='original_du.conf',
        tofile='modified_du.conf'
    ))
    return new_conf, diff


# 主函式

def adjust_ssb_from_query(du_conf: str, query: str) -> Tuple[str, str]:
    freq_hz = parse_freq_from_query(query)
    arfcn = freq_to_arfcn(freq_hz)
    new_conf, diff = modify_absolute_frequency_ssb(du_conf, arfcn)
    return new_conf, diff


In [5]:
# 使用方式範例
if __name__ == "__main__":
    # 模擬輸入（可以替換為實際讀取檔案）
    with open("du.conf", "r") as f:
        original_du_conf = f.read()

    query = "請將 SSB 頻率設定為 3.6GHz"

    new_conf, diff = adjust_ssb_from_query(original_du_conf, query)

    with open("du_modified.conf", "w") as f:
        f.write(new_conf)

    with open("du_diff.patch", "w") as f:
        f.write(diff)

    print("✅ 修改完成，結果已輸出 du_modified.conf 與 du_diff.patch")


FileNotFoundError: [Errno 2] No such file or directory: 'du.conf'